For NUS:
    keep:
        "department"
        "description"
        "faculty"
        "moduleCode"
        "title"
        
    ignore:
        "gradingBasisDescription"
        "moduleCredit"
        "semesterData"
        "workload"


For NTU:
    keep:
        "code" 
        "title" 
        "description" 
        "departmentMaintaining"

    ignore:
        "academicUnits"
        "examSchedule" 
        "gradeType" 
        "prerequisites" 
        "notAvailableToProgramme" (eg not available to EEE)
        "url" 
        "details" 


For SUTD:
    keep:
        "code"
        "title" 
        "course_type" (core, electives, etc)
        "description" 
        *missing department

    ignore:
        "terms" 
        "credits" 
        "url" 
        "description_found" (T/F)


extract skills from the descriptions

courses:
|university| department| faculty| code| title| course_type| description| skills|

jobs:
    keep: 
        "title"
        "description"
        "minimumYearsExperience"
        "shiftPattern"
        "skills" (keep everything in the skills section)
            "uuid": "11ff9f68afb6b8b5b8eda218d7c83a65",
            "confidence": null,
            "isKeySkill": null
        "otherRequirements"
        "ssocCode" (job industry/sector)
        "occupationId"
        "ssocVersion"
        "workingHours"
        "numberOfVacancies"
        "categories"
            "id"
            "category"
        "employmentTypes" 
            "id" 
            "employmentType" 
        "positionLevels"
            "id": 11,
            "position" 
        "ssecEqa" 
            (Singapore Standard Educational Classification (SSEC) Educational Qualification Assessment code, indicating the education level. 
            eg "69" typically corresponds to a Bachelor's Degree with Honours.)
        "ssecFos" 
            (Field of Study code, 
            eg "0212" corresponds to "Music and Performing Arts")
        "ssicCode" 
            (It classifies the type of industry/business sector the company operates in.)
            (SSOC = the job classification (Laboratory Technician, Finance Manager, etc.)
            (SSIC = the company/industry classification (Manufacturing, Education, etc.))
        "ssicCode2020" 
         "salary": {
                    "maximum": 3500,
                    "minimum": 2500,
                    "type": {
                    "id": 4,
                    "salaryType": "Monthly"
                    }
        "jobTitles"

    keep but tbc: (if all NA/nulls -> drop)
        "schemes" 
        "flexibleWorkArrangements"

    ignore:
        "uuid" (post id? user id?)
        "sourceCode" 
        "psdUrl"
        "status"
        "postedCompany" 
            "uen" 
                ("uen" stands for Unique Entity Number, a unique identifier assigned to all business entities (companies, organizations, partnerships) registered in Singapore. In the example, "52875677W" is the UEN for SYMPHONY MUSIC SCHOOL. It's used to uniquely identify and track companies in Singapore's business registry.)
            "description" (company description)
            "name" (company name)
        "employeeCount" 
        "companyUrl"
        "lastSyncDate": "2026-01-30T02:57:44.000Z",
        "_links"
        "logoFileName" 
        "logoUploadPath" 
        "responsiveEmployer": {
      "isResponsive": false
    }



# NUS

## Load Data

In [73]:
# import necessary libraries
import json5
import pandas as pd
from pathlib import Path

In [62]:
# load output NUSModsInfo.json file
nus_file_path = Path("../../data/NUSModsInfo.json")

with open(nus_file_path, "r", encoding="utf-8") as f:
    nus_data = json5.load(f)

In [63]:
# Extract relevant fields
nus_fields = ["department", "description", "faculty", "moduleCode", "title"]

nus_filtered_data = [
    {key: item.get(key) for key in nus_fields}
    for item in nus_data
]

In [64]:
# Create DataFrame
nus_df = pd.DataFrame(nus_filtered_data)

# Preview
nus_df.head()

,department,description,faculty,moduleCode,title
0,NUS Medicine Dean's Office,Leadership is fundamental to the success of in...,Yong Loo Lin Sch of Medicine,ABM5001,Leadership in Biomedicine
1,NUS Medicine Dean's Office,This course serves as a concept-based introduc...,Yong Loo Lin Sch of Medicine,ABM5002,Advanced Biostatistics for Research
2,NUS Medicine Dean's Office,This course will furnish students with a thoro...,Yong Loo Lin Sch of Medicine,ABM5003,Biomedical Innovation & Enterprise
3,NUS Medicine Dean's Office,This course encompasses research projects rele...,Yong Loo Lin Sch of Medicine,ABM5004,Capstone Project
4,NUS Medicine Dean's Office,Advanced immunological applications play impor...,Yong Loo Lin Sch of Medicine,ABM5101,Applied Immunology


## Data Cleaning

### Data Exploration

In [65]:
# check for missing values
print(nus_df.isnull().sum())

department     103
description      0
faculty          0
moduleCode       0
title            0
dtype: int64


In [66]:
nus_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20517 entries, 0 to 20516
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   department   20414 non-null  str  
 1   description  20517 non-null  str  
 2   faculty      20517 non-null  str  
 3   moduleCode   20517 non-null  str  
 4   title        20517 non-null  str  
dtypes: str(5)
memory usage: 801.6 KB


In [67]:
# unique values in department and faculty
print("Unique departments:", nus_df["department"].nunique())
print("Unique faculties:", nus_df["faculty"].nunique()) 

Unique departments: 109
Unique faculties: 25


In [68]:
# get unique faculties 
unique_faculties = nus_df["faculty"].unique()
print("Unique faculties:", unique_faculties)

Unique faculties: <StringArray>
[     'Yong Loo Lin Sch of Medicine', 'College of Design and Engineering',
               'NUS Business School',           'Arts and Social Science',
                               'NUS',                         'Computing',
                               'Law',       'Cont and Lifelong Education',
                           'Science',     'Non-Faculty-based Departments',
                         'Dentistry',         'YST Conservatory of Music',
      'Multi Disciplinary Programme',                           'NUS-ISS',
               'Residential College',    'Temasek Defence Sys. Institute',
         'Risk Management Institute',                       'NUS College',
       'SSH School of Public Health',           'Duke-NUS Medical School',
               'NUS Graduate School',           'Logistics Inst-Asia Pac',
    'Mechanobiology Institute (MBI)',       'LKY School of Public Policy',
                  'Yale-NUS College']
Length: 25, dtype: str


In [69]:
# faculty level data distribution
faculty_counts = nus_df["faculty"].value_counts()
print(faculty_counts)

faculty
Arts and Social Science              5679
Law                                  2878
College of Design and Engineering    2823
Science                              1695
NUS Business School                  1642
Yale-NUS College                     1485
Computing                             541
Yong Loo Lin Sch of Medicine          517
Cont and Lifelong Education           425
Duke-NUS Medical School               416
LKY School of Public Policy           377
YST Conservatory of Music             373
Non-Faculty-based Departments         366
NUS College                           337
Residential College                   261
SSH School of Public Health           208
NUS                                   154
NUS-ISS                                82
NUS Graduate School                    65
Dentistry                              59
Temasek Defence Sys. Institute         51
Risk Management Institute              48
Multi Disciplinary Programme           15
Mechanobiology Institute (

### Filter for undergraduate courses only

In [70]:
# Keep only undergraduate level courses: codes where the first digit after letters is 0-4
nus_df = nus_df[nus_df['moduleCode'].str.match(r'^[A-Z]+[0-4]')]

nus_df

,department,description,faculty,moduleCode,title
27,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701,Accounting for Decision Makers
28,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701A,Accounting for Decision Makers
29,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701B,Accounting for Decision Makers
30,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701C,Accounting for Decision Makers
31,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701D,Accounting for Decision Makers
...,...,...,...,...,...
20512,Biological Sciences,In addition to having an academic science foun...,Science,ZB3312,Enhanced Undergraduate Professional Internship...
20513,Biological Sciences,In addition to having an academic science foun...,Science,ZB3313,Undergraduate Professional Internship Programm...
20514,Biological Sciences,This is a seminar-style course based on the li...,Science,ZB4171,Advanced Topics in Bioinformatics
20515,Biological Sciences,Not Available,Science,ZB4199,Honours Project in Computational Biology


In [71]:
# faculty level data distribution
faculty_counts = nus_df["faculty"].value_counts()
print(faculty_counts)

faculty
Arts and Social Science              4631
Yale-NUS College                     1485
College of Design and Engineering    1374
Science                              1172
NUS Business School                   796
Law                                   783
Cont and Lifelong Education           403
Computing                             350
YST Conservatory of Music             337
NUS College                           337
Non-Faculty-based Departments         333
Residential College                   261
Yong Loo Lin Sch of Medicine          174
NUS                                   122
SSH School of Public Health            61
Dentistry                              50
NUS-ISS                                19
Multi Disciplinary Programme           15
Duke-NUS Medical School                 3
Temasek Defence Sys. Institute          2
Name: count, dtype: int64


In [72]:
# filter out post graduate level courses: faculty is for post graduate level courses, which are not relevant to our analysis
target_faculties = [
    "Temasek Defence Sys. Institute",
     "Cont and Lifelong Education",
     "Duke-NUS Medical School"
]

filtered_nus_df = nus_df[~nus_df["faculty"].isin(target_faculties)]

### Handling NA Values

In [60]:
# count data with null description
null_description_count = filtered_nus_df["description"].isnull().sum()
print("Number of courses with null description:", null_description_count)

Number of courses with null description: 0


In [74]:
# Distribution of description data
description_counts = filtered_nus_df["description"].value_counts()
print(description_counts)

description
Not Available                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               3664
UTOP aims to train undergraduates to acquire and promulgate “teaching” skills. When pursuing such traineeship in teaching under UTOP, students can develop their teaching skills in a more systematic manner, and become be

some short descriptions such as: NIL, Exchange Course - YUS (1 unit), Not Applicable 
are not meaningful for analysis. we would remove those data

In [76]:
# Keep only rows where description has 10 or more words
filtered_nus_df = filtered_nus_df[filtered_nus_df['description'].str.split().str.len() >= 10]

In [78]:
filtered_nus_df.info()

<class 'pandas.DataFrame'>
Index: 8499 entries, 27 to 20516
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   department   8499 non-null   str  
 1   description  8499 non-null   str  
 2   faculty      8499 non-null   str  
 3   moduleCode   8499 non-null   str  
 4   title        8499 non-null   str  
dtypes: str(5)
memory usage: 398.4 KB


there is no more NA data in the dataset

## Data Transformation